In [ ]:
import torch
from torch import nn
import torch.nn.functional as F

import matplotlib.pyplot as plt

In [ ]:
with open("pokemon_names.txt", "r") as f:
    text = f.read()

vocab = sorted(set(text))
vocab.append("<>")
print(''.join(vocab))

vocab_size = len(vocab)
text = text.splitlines()

In [ ]:
tokens, rtok = {}, {}
for i, ch in enumerate(vocab):
    tokens[ch] = i
    rtok[i] = ch

In [ ]:
freq = {}
for i in range(len(text)):
    chars = ["<>"] + list(text[i]) + ["<>"]
    for ch1, ch2 in zip(chars, chars[1:]):
        pair = (ch1, ch2)
        freq[pair] = freq.get(pair, 0) + 1
sp = sorted(freq.items(), key=lambda k: (-freq[k[0]], k[0]))
print(sp)

In [ ]:
prob_dist = torch.zeros(vocab_size, vocab_size)

for ((ch1, ch2), freq) in sp:
    idx1 = tokens[ch1]
    idx2 = tokens[ch2]
    prob_dist[idx1, idx2] = freq
    
print(prob_dist)

In [ ]:
# fg = prob_dist.numpy()

# plt.figure(figsize=(16, 16))
# plt.imshow(fg, cmap='viridis')

# # optional colorbar + labels
# plt.colorbar(label='Value Intensity')
# plt.title('Bigram Probability Heatmap')
# plt.xlabel('Next char')
# plt.ylabel('Current char')

# # annotate every cell
# for i in range(62):
#     for j in range(62):
#         chstr = itos[i] + itos[j]
        
#         plt.text(j, i, chstr,
#                  ha="center", va="bottom", color='gray', fontsize=6)
        
#         plt.text(j, i, f"{fg[i, j]:.2f}",
#                  ha="center", va="top", color='gray', fontsize=6)

# plt.axis('off')
# plt.show()

In [ ]:
tok = int(torch.rand(1).item() * vocab_size)

x, y = prob_dist.shape
for i in range(x):
    prob_dist[i] = prob_dist[i] / (prob_dist[i].sum() + 0.0001)

In [ ]:
for i in range(5):
    likelihood = 1
    dist = prob_dist[tokens["<>"]]
    st = torch.multinomial(input=dist, num_samples=1, replacement=True)
    while True:
        nc = rtok[st.item()]
        neg_log = torch.log(dist[st])
        likelihood += neg_log
        if(nc == "<>"):
            break
        print(f"{nc}", end="")
        dist = prob_dist[tokens[nc]]
        st = torch.multinomial(input=dist, num_samples=1, replacement=True)

    print(f"{likelihood.item():>10.3f}")

In [ ]:
""" Defining the model """

class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Sequential(
            nn.Linear(vocab_size, vocab_size),
            nn.ReLU(),
            nn.Linear(vocab_size, vocab_size),
        )

    def forward(self, x):
        logits = self.linear(x)
        return logits
    
model = NeuralNetwork().to("cpu")
optimizer = torch.optim.SGD(model.parameters(), lr=1e-2)
print(model)

In [ ]:
for i in range(10):
    likelihood = 1
    dist = prob_dist[tokens["<>"]]
    st = torch.multinomial(input=dist, num_samples=1, replacement=True)
    pred = st.item()
    for i in range(20):
        nc = rtok[pred]
        neg_log = torch.log(dist[pred])
        likelihood += neg_log
        if(nc == "<>"):
            break
        print(f"{nc}", end="")
        dist = prob_dist[tokens[nc]]
  
        " Training the model "
        st = model(dist)
        pred = torch.argmax(st).item()

    print(f"{likelihood.item():>10.3f}")

In [ ]:
def get_train_data(name):
    train_X, train_Y = [], []
   
    chars = ["<>"] + [ch for ch in name] + ["<>"]
    for ch1, ch2 in zip(chars, chars[1:]):
        train_X.append(ch1)
        train_Y.append(ch2)
        
    return train_X, train_Y

get_train_data("Bulbasaur")

In [ ]:
num_epochs = 100

for epoch in range(num_epochs):
    model.train()
    total_loss = 0.0
    count = 0

    for name in text:
        train_X, train_Y = get_train_data(name)
        for ch_X, ch_Y in zip(train_X, train_Y):
            x = F.one_hot(torch.tensor(tokens[ch_X]), num_classes=vocab_size).float().unsqueeze(0)
            target = torch.tensor([tokens[ch_Y]], dtype=torch.long)  # shape [1]
            logits = model(x)  # shape [1, vocab_size]
            loss = F.cross_entropy(logits, target)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            count += 1

    ### LOGGING
    print(f"Epoch {epoch+1}: avg loss = {total_loss / count:.4f}")

model.eval()


In [753]:
for i in range(20):
    likelihood = 1
    dist = prob_dist[tokens["<>"]]
    st = torch.multinomial(input=dist, num_samples=1, replacement=True)
    pred = st.item()
    for i in range(20):
        nc = rtok[pred]
        neg_log = torch.log(dist[pred])
        likelihood += neg_log
        if(nc == "<>"):
            break
        print(f"{nc}", end="")
        dist = prob_dist[tokens[nc]]
  
        " Training the model "
        st = model(dist)
        pred = torch.argmax(st).item()

    print(f"{likelihood.item():>10.3f}")

E      -inf
F      -inf
S      -inf
P      -inf
Ln      -inf
S      -inf
S      -inf
K      -inf
G      -inf
K      -inf
S      -inf
R      -inf
D      -inf
S      -inf
Vl      -inf
M      -inf
K      -inf
R      -inf
R      -inf
P      -inf
